# Milestone 013 — Accessible does not imply used

**Research period:** September 8–9, 2026  
**Historical anchor:** #328

The revision-branch benchmark gave me a task where preserving a distinction actually mattered. I could now return to the compact unit-hypersphere state and ask a much more precise question: **was the information gone, or was the model simply failing to use information that was still there?**

Those possibilities require different fixes.

If the representation has destroyed the distinction, changing the final readout can't recover it. If the distinction remains accessible, retraining or replacing the representation may be unnecessary; the bottleneck could instead be the small piece of computation that turns the internal state into a prediction.

I therefore froze the representation and tested what a deliberately simple new consumer could recover from it.


## Affine probing

The first probe was intentionally small. Given a frozen hidden state `h`, it computes

`z = Wh + b`

and converts those scores into probabilities.

There are no additional representation layers and no nonlinear feature extractor. If this probe succeeds on held-out data, the relevant distinction must already be accessible through a simple affine readout of the frozen state.

> **Sticky note — affine map:** a linear transformation followed by an added bias. In this case, the probe can reweight and combine existing coordinates, but it can't build a deep new representation of its own. [Reference →](../reference/glossary.md#affine-map)

The result was much stronger than I expected. On the frozen natural-source branch benchmark, the affine-softmax probe improved evaluation log loss over the constant baseline by about **0.738 nat**, with a one-sided lower confidence bound of about **0.678 nat**.

I repeated the analysis using tangent coordinates so that the probe couldn't rely on an obvious radial explanation, and the conclusion survived. The selectivity controls also passed.

The model's own frozen consumer told a very different story: it recovered only a small fraction of the predictive gain that the probe showed was available.

I classified the result as **ACCESSIBLE-BUT-UNDERUSED**.

The distinction matters. The compact representation hadn't simply erased the branch information. A small separately trained readout could find a strong signal in the state; the native model was leaving most of that signal unused.

## Why I don't trust a successful probe by itself

A sufficiently flexible diagnostic can manufacture an impressive-looking result from weak structure, memorize quirks of the dataset, exploit leakage, or succeed simply because its analysis pipeline is powerful enough.

So the useful question wasn't just whether the probe worked. It was whether it worked **selectively**.

I attacked the result with label shuffles, representation shuffles, matched random features, random rotations, fixed page groups, and tests of how efficiently the probe learned the distinction as data increased. Each control targeted a different alternative explanation: accidental label structure, memorization, coordinate artifacts, source leakage, or an analysis procedure that would find signal almost anywhere.

> **Sticky note — selectivity:** a diagnostic is selective when it succeeds where the intended structure is present and fails appropriately when that structure is deliberately destroyed. [Reference →](../reference/glossary.md#selectivity)

The tangent-coordinate analysis served a related purpose. The earlier hypersphere work had made radial versus tangent structure an obvious possible confound, so I wanted to know whether the branch signal survived when expressed intrinsically along the sphere rather than through the original ambient coordinates.

It did.

## Research context

Linear probing is widely used in representation learning because it asks a deliberately limited question: can a simple downstream model recover some property from a frozen representation?

That limitation is also why probe results need careful interpretation. Research on probing classifiers has repeatedly pointed out that a powerful probe can obscure the distinction between information encoded simply in a representation and information reconstructed through the probe's own capacity. Selectivity controls and deliberately restricted probe families are ways of keeping that interpretation narrower.

For this project, the probe wasn't intended to show that the representation was globally sufficient or that the original model had learned to use the branch distinction. Its role was diagnostic: distinguish **information absent from the state** from **information accessible to a small external consumer**.

That was enough to change the next experiment.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng=np.random.default_rng(7); n,d=4000,24
y=rng.integers(0,2,n); h=rng.normal(size=(n,d))
v=rng.normal(size=d); v/=np.linalg.norm(v); h+=(2*y[:,None]-1)*0.6*v
Xt,Xe,yt,ye=train_test_split(h,y,test_size=.3,random_state=1,stratify=y)
p=LogisticRegression(max_iter=1000).fit(Xt,yt)
print('held-out probe accuracy:',p.score(Xe,ye))


## Accessibility versus utilization

There are now at least three separate questions:

1. **Is task-relevant information present in the representation?**
2. **Can a bounded consumer recover it?**
3. **Does the model's native consumer actually use it for the task we care about?**

The branch probe gave strong evidence for the second question and therefore against the simplest version of “the compact state destroyed the distinction.” The weak native result showed that the third question could have a very different answer.

This also explains why I didn't respond by training a larger representation model. The representation had already passed the cheaper accessibility test. The next intervention belonged downstream.

## What this does not show

Probe success doesn't establish that the native model causally uses the information. The probe is separately trained and can exploit a signal that the original consumer ignores.

It also doesn't establish that the frozen representation contains everything needed for language modeling, or even everything needed for the full next-byte task. The revision benchmark tests one bounded natural-source distinction.

Most importantly, predicting which branch a continuation belongs to is much easier than predicting the actual next byte from all **256 possible byte values**. A strong binary or branch-level signal may still be difficult to turn into better full-distribution predictions.

The next experiment therefore keeps the representation frozen again and moves one step closer to the real task: train a ladder of very small consumers on the actual 256-way next-byte objective and measure how much of the accessible signal can be converted into useful prediction.
